# GroupDNA — WhatsApp Group Behavioural Analytics

**Student name:** Add your name  
**Roll number:** Add your roll number  
**Batch:** Add your batch  
**Date:** 22 November 2026

> AI-assisted project scaffold: review each section, understand the logic, and personalize the wording and code before submission as required by the project brief.


## Project introduction

GroupDNA turns a WhatsApp text export into a readable summary of group activity, word use, response patterns, silent periods, and behavioural archetypes. This notebook is written for a student learning Python: each section introduces the calculation and keeps the implementation in functions and ordinary dictionaries.

## Objective and scope

The academic notebook follows the PDF's restrictions: Python fundamentals, file I/O, `datetime`, and NumPy. It does not use pandas, plotting packages, `collections.Counter`, regex, or pre-built chat parsers. The supplied file is a synthetic practice export. Its results describe that practice file only; they do not represent a real group or the brief's original reference counts.


## 4. Library imports

NumPy is used for the required participant-by-hour matrix. `datetime` converts timestamps and measures elapsed time.

In [ ]:
# AI-assisted: import and parsing structure reviewed for this student project.
"""GroupDNA analysis built with Python fundamentals and NumPy only.

The module intentionally avoids pandas, plotting libraries, regex, Counter,
and pre-built WhatsApp parsers so it can support the academic notebook.
"""

from datetime import datetime, timedelta

import numpy as np


DATE_FORMAT = "%d/%m/%y, %H:%M"
MEDIA_MARKER = "<Media omitted>"
DELETED_MARKERS = ("This message was deleted", "You deleted this message")
STOP_WORDS = {
    "i", "is", "the", "a", "and", "or", "to", "of", "in", "on", "for",
    "you", "it", "we", "are", "was", "be", "this", "that", "with", "my",
    "me", "your", "they", "he", "she", "am", "an", "at", "as", "but",
    "so", "if", "do", "not", "have", "has", "had", "from", "by", "our",
}
CARING_PHRASES = (
    "okay", "safe", "eat", "sleep", "take care", "are you", "please",
    "reminder", "drink water", "don't forget", "dont forget",
)
COMEDY_WORDS = ("lol", "lmao", "haha", "rofl", "lmfao")
FOOD_WORDS = ("mess", "food", "tea", "chai", "lunch", "dinner", "breakfast", "kadai", "canteen")
ARCHETYPE_NAMES = (
    "The Spammer", "The Group Mom", "The Night Owl", "The Storyteller",
    "The Drama Queen", "The Ghost", "The Comedian", "The Question Master",
)


## 5. Dataset upload

In Colab, upload the `.txt` file using the Files panel and set `CHAT_FILE` to its path. When the file is next to this notebook, the default name works.

In [ ]:
CHAT_FILE = '/content/GroupDNA_synthetic_hostel_bois_practice.txt'


## 6. Initial dataset inspection

Inspect file existence, line count, and a small sample before parsing. The complete chat is not printed.

In [ ]:
with open(CHAT_FILE, 'r', encoding='utf-8') as chat_file:
    raw_lines = chat_file.readlines()

print('Raw lines:', len(raw_lines))
print('First five raw lines:')
for line in raw_lines[:5]:
    print(line.rstrip())
print('Last five raw lines:')
for line in raw_lines[-5:]:
    print(line.rstrip())


Raw lines: 3177
First five raw lines:
01/04/24, 00:10 - Aman: enna comedy da
01/04/24, 01:13 - Priya created group "Hostel Bois 4ever"
01/04/24, 01:14 - Priya added you
01/04/24, 01:15 - Aman: one small doubt about the project
01/04/24, 01:36 - Aman: did you finish the work?
Last five raw lines:
30/05/24, 21:09 - Priya: reach safely and message here
30/05/24, 22:13 - Priya: everyone had dinner
30/05/24, 22:23 - Rahul: mess ka food aaj kaisa tha
30/05/24, 22:35 - Neha: I CANNOT BELIEVE THIS!!!
30/05/24, 23:15 - Neha: WHY DOES THIS ALWAYS HAPPEN TO US!!


## Feature 1 — WhatsApp chat parser

A new timestamp starts an entry. Lines without a timestamp continue the previous entry. System entries are counted separately; media and deleted entries remain participant activity, but are excluded from text analysis.

In [ ]:
# AI-assisted: parser supports the documented WhatsApp export edge cases.
def looks_like_timestamp(line):
    """Check the WhatsApp date/time prefix without regular expressions."""
    if len(line) < 18 or line[2:3] != "/" or line[5:6] != "/":
        return False
    if line[8:10] != ", " or line[12:13] != ":" or line[15:18] != " - ":
        return False
    return (line[0:2] + line[3:5] + line[6:8] + line[10:12] + line[13:15]).isdigit()


def parse_chat_file(file_path):
    """Read an export and retain participant, system, media and deleted entries."""
    with open(file_path, "r", encoding="utf-8") as chat_file:
        return parse_chat_lines(chat_file)


def parse_chat_lines(lines):
    """Parse lines directly; this also makes small parser checks easy to read."""
    records = []
    counters = {"system": 0, "media": 0, "deleted": 0, "empty": 0, "continuation": 0}
    for raw_line in lines:
        line = raw_line.rstrip("\r\n")
        if not line.strip():
            counters["empty"] += 1
            continue
        if not looks_like_timestamp(line):
            if records:
                records[-1]["text"] += "\n" + line
                counters["continuation"] += 1
            continue
        timestamp_text, remainder = line.split(" - ", 1)
        if ": " in remainder:
            sender, message_text = remainder.split(": ", 1)
        else:
            sender, message_text = "System", remainder
            counters["system"] += 1
        timestamp = datetime.strptime(timestamp_text, DATE_FORMAT)
        record = {"timestamp": timestamp, "timestamp_text": timestamp_text,
                  "sender": sender, "text": message_text, "kind": "participant"}
        if sender == "System":
            record["kind"] = "system"
        elif MEDIA_MARKER in message_text:
            record["kind"] = "media"
            counters["media"] += 1
        elif any(marker in message_text for marker in DELETED_MARKERS):
            record["kind"] = "deleted"
            counters["deleted"] += 1
        records.append(record)
    return records, counters


def get_participant_records(records):
    return [record for record in records if record["kind"] != "system"]


def get_analyzable_records(records):
    return [record for record in records if record["kind"] == "participant"]


def count_words(message_text):
    return len(message_text.split())

records, parser_counters = parse_chat_file(CHAT_FILE)
participant_records = get_participant_records(records)
text_records = get_analyzable_records(records)
participants = sorted({record['sender'] for record in participant_records})
print(f'Parsed {len(records):,} total entries: {len(participant_records):,} participant entries, {len(text_records):,} analyzable text messages, {len(participants)} participants.')
print('System / media / deleted:', parser_counters['system'], '/', parser_counters['media'], '/', parser_counters['deleted'])
print('First five parsed records:')
for record in records[:5]:
    print(record)
print('Last five parsed records:')
for record in records[-5:]:
    print(record)


Parsed 3,177 total entries: 3,173 participant entries, 3,128 analyzable text messages, 6 participants.
System / media / deleted: 4 / 30 / 15
First five parsed records:
{'timestamp': datetime.datetime(2024, 4, 1, 0, 10), 'timestamp_text': '01/04/24, 00:10', 'sender': 'Aman', 'text': 'enna comedy da', 'kind': 'participant'}
{'timestamp': datetime.datetime(2024, 4, 1, 1, 13), 'timestamp_text': '01/04/24, 01:13', 'sender': 'System', 'text': 'Priya created group "Hostel Bois 4ever"', 'kind': 'system'}
{'timestamp': datetime.datetime(2024, 4, 1, 1, 14), 'timestamp_text': '01/04/24, 01:14', 'sender': 'System', 'text': 'Priya added you', 'kind': 'system'}
{'timestamp': datetime.datetime(2024, 4, 1, 1, 15), 'timestamp_text': '01/04/24, 01:15', 'sender': 'Aman', 'text': 'one small doubt about the project', 'kind': 'participant'}
{'timestamp': datetime.datetime(2024, 4, 1, 1, 36), 'timestamp_text': '01/04/24, 01:36', 'sender': 'Aman', 'text': 'did you finish the work?', 'kind': 'participant'}
Las

## Feature 2 — Group overview

Message activity includes media and deleted entries. Word totals and average message lengths use analyzable text messages. The observed duration counts both the first and last calendar date.

In [ ]:
# AI-assisted: overview calculations use the records parsed above.
def build_statistics(records):
    """Calculate all dashboard/report metrics from parsed message records."""
    records = sorted(records, key=lambda record: record["timestamp"])
    participant_records = get_participant_records(records)
    analyzable = get_analyzable_records(records)
    participants = sorted({record["sender"] for record in participant_records})
    all_timestamps = [record["timestamp"] for record in records]
    if not participants or not all_timestamps:
        return {
            "group_name": "Unknown WhatsApp Group", "participants": [], "participant_stats": {},
            "total_entries": len(records), "total_messages": 0, "analyzable_messages": len(analyzable),
            "total_words": 0, "start": None, "end": None, "duration_days": 0,
            "daily_counts": {}, "hourly_counts": [0] * 24, "activity_matrix": [],
            "activity_hours": [], "top_words": [], "response_minutes": {},
            "silent_streaks": {}, "archetypes": {}, "date_labels": [],
        }

    first_time = min(all_timestamps)
    last_time = max(all_timestamps)
    start_day = first_time.date()
    end_day = last_time.date()
    duration_days = (end_day - start_day).days + 1
    day_count = {}
    hour_count = [0] * 24
    message_count = {name: 0 for name in participants}
    media_count = {name: 0 for name in participants}
    deleted_count = {name: 0 for name in participants}
    word_count_by_person = {name: 0 for name in participants}
    word_frequency = {}
    word_frequency_by_person = {name: {} for name in participants}
    response_gaps = {name: [] for name in participants}
    matrix = np.zeros((len(participants), 24), dtype=int)
    participant_index = {name: index for index, name in enumerate(participants)}
    previous_participant = None
    previous_timestamp = None
    total_words = 0
    group_name = "Unknown WhatsApp Group"

    for record in records:
        if record["kind"] == "system":
            if 'created group "' in record["text"]:
                group_name = record["text"].split('created group "', 1)[1].split('"', 1)[0]
            continue
        sender = record["sender"]
        message_time = record["timestamp"]
        message_count[sender] += 1
        day_key = message_time.strftime("%Y-%m-%d")
        day_count[day_key] = day_count.get(day_key, 0) + 1
        hour_count[message_time.hour] += 1
        matrix[participant_index[sender], message_time.hour] += 1
        if record["kind"] == "media":
            media_count[sender] += 1
        elif record["kind"] == "deleted":
            deleted_count[sender] += 1
        else:
            words = record["text"].split()
            word_count_by_person[sender] += len(words)
            total_words += len(words)
            for raw_word in words:
                word = raw_word.lower().strip("!\"#$%&'()*+,-./:;<=>?@[\\]^_`{|}~“”‘’…")
                if word and word not in STOP_WORDS:
                    word_frequency[word] = word_frequency.get(word, 0) + 1
                    person_words = word_frequency_by_person[sender]
                    person_words[word] = person_words.get(word, 0) + 1
        if previous_participant is not None and sender != previous_participant:
            gap_minutes = (message_time - previous_timestamp).total_seconds() / 60
            if gap_minutes >= 0:
                response_gaps[sender].append(gap_minutes)
        previous_participant = sender
        previous_timestamp = message_time

    # Keep every date in the observed range for daily charts and silent-day logic.
    date_labels = [(start_day + timedelta(days=offset)).isoformat() for offset in range(duration_days)]
    daily_counts = {day: day_count.get(day, 0) for day in date_labels}
    participant_stats = {}
    silent_streaks = {}
    for name in participants:
        person_records = [record for record in participant_records if record["sender"] == name]
        person_days = {record["timestamp"].date() for record in person_records}
        best_length = 0
        best_start = None
        current_length = 0
        current_start = None
        for day_offset in range(duration_days):
            day = start_day + timedelta(days=day_offset)
            if day not in person_days:
                if current_length == 0:
                    current_start = day
                current_length += 1
                if current_length > best_length:
                    best_length = current_length
                    best_start = current_start
            else:
                current_length = 0
                current_start = None
        average_length = word_count_by_person[name] / max(1, sum(
            1 for record in analyzable if record["sender"] == name
        ))
        participant_stats[name] = {
            "messages": message_count[name],
            "words": word_count_by_person[name],
            "average_words": average_length,
            "contribution_percent": 100 * message_count[name] / max(1, len(participant_records)),
            "media": media_count[name], "deleted": deleted_count[name],
            "average_response_minutes": (sum(response_gaps[name]) / len(response_gaps[name])) if response_gaps[name] else None,
            "response_count": len(response_gaps[name]),
            "silent_days": duration_days - len(person_days),
            "silent_percent": 100 * (duration_days - len(person_days)) / duration_days,
            "longest_silent_days": best_length,
            "silent_start": best_start.isoformat() if best_start else None,
            "silent_end": (best_start + timedelta(days=best_length - 1)).isoformat() if best_start and best_length else None,
            "active_days": len(person_days),
        }
        silent_streaks[name] = participant_stats[name]["longest_silent_days"]

    top_words = sorted(word_frequency.items(), key=lambda pair: (-pair[1], pair[0]))[:10]
    hourly_max = max(hour_count) if hour_count else 0
    busiest_hour = hour_count.index(hourly_max) if hourly_max else None
    busiest_day = max(daily_counts, key=daily_counts.get) if daily_counts else None
    archetypes = detect_archetypes(participant_records, participants, participant_stats) if "detect_archetypes" in globals() else {}
    return {
        "group_name": group_name, "participants": participants, "participant_stats": participant_stats,
        "total_entries": len(records), "total_messages": len(participant_records),
        "analyzable_messages": len(analyzable), "system_messages": sum(1 for r in records if r["kind"] == "system"),
        "media_messages": sum(media_count.values()), "deleted_messages": sum(deleted_count.values()),
        "total_words": total_words, "start": first_time.isoformat(), "end": last_time.isoformat(),
        "duration_days": duration_days, "daily_counts": daily_counts, "hourly_counts": hour_count,
        "busiest_day": busiest_day, "busiest_day_count": daily_counts.get(busiest_day, 0),
        "busiest_hour": busiest_hour, "busiest_hour_count": hourly_max,
        "average_hourly_activity": [count / duration_days for count in hour_count],
        "activity_matrix": matrix.tolist(), "activity_hours": list(range(24)),
        "top_words": top_words, "word_frequency_by_person": word_frequency_by_person,
        "response_minutes": response_gaps, "silent_streaks": silent_streaks,
        "archetypes": archetypes, "date_labels": date_labels,
    }

stats = build_statistics(records)
print('Group:', stats['group_name'])
print('Messages from participants:', f"{stats['total_messages']:,}")
print('Analyzable text messages:', f"{stats['analyzable_messages']:,}")
print('Participants:', len(stats['participants']))
print('Period:', stats['start'], 'to', stats['end'], f"({stats['duration_days']} days)")
print('Total words in analyzable messages:', f"{stats['total_words']:,}")
print('\nMESSAGES PER PARTICIPANT')
for name in sorted(stats['participants'], key=lambda person: (-stats['participant_stats'][person]['messages'], person)):
    person = stats['participant_stats'][name]
    print(f"  {name:<14} {person['messages']:>5,}  ({person['contribution_percent']:5.1f}%)  {person['words']:>5,} words  avg {person['average_words']:.1f} words/text")


Group: Hostel Bois 4ever
Messages from participants: 3,173
Analyzable text messages: 3,128
Participants: 6
Period: 2024-04-01T00:10:00 to 2024-05-30T23:15:00 (60 days)
Total words in analyzable messages: 15,519

MESSAGES PER PARTICIPANT
  Rahul          1,177  ( 37.1%)  4,217 words  avg 3.6 words/text
  Priya            580  ( 18.3%)  2,641 words  avg 4.6 words/text
  Neha             510  ( 16.1%)  2,358 words  avg 4.7 words/text
  Aman             476  ( 15.0%)  1,930 words  avg 4.1 words/text
  Karan            389  ( 12.3%)  4,225 words  avg 11.0 words/text
  Vikas             41  (  1.3%)    148 words  avg 4.4 words/text


## Feature 3 — Most active day and hour

Daily totals include every participant entry. Hourly averages divide each hour's count by the number of observed calendar days.

In [ ]:
busiest_day = stats['busiest_day']
hour = stats['busiest_hour']
print('Busiest day:', busiest_day, f"({stats['busiest_day_count']} messages)")
print('Busiest hour:', f"{hour:02d}:00–{(hour + 1) % 24:02d}:00", f"({stats['busiest_hour_count']} messages; {stats['average_hourly_activity'][hour]:.1f} per day)")


Busiest day: 2024-04-19 (66 messages)
Busiest hour: 13:00–14:00 (197 messages; 3.3 per day)


## Feature 4 — NumPy activity heatmap

Rows are dynamically created for the participants found in the export. Each of the 24 columns represents one hour. Cell shading is based on that participant's busiest hour, using the PDF's percentage bands.

In [ ]:
activity_matrix = np.array(stats['activity_matrix'], dtype=int)
print('NumPy matrix shape:', activity_matrix.shape)
print('Hour  ' + ' '.join(f'{hour:02d}' for hour in range(24)))
for row_index, name in enumerate(stats['participants']):
    row = activity_matrix[row_index]
    row_max = int(np.max(row))
    symbols = []
    for value in row:
        share = value / row_max if row_max else 0
        if value == 0 or share < 0.25:
            symbols.append('.')
        elif share < 0.50:
            symbols.append('░')
        elif share < 0.75:
            symbols.append('▒')
        else:
            symbols.append('█')
    print(f'{name:<14}' + ' '.join(symbols))
print('Hourly totals:', np.sum(activity_matrix, axis=0))


NumPy matrix shape: (6, 24)
Hour  00 01 02 03 04 05 06 07 08 09 10 11 12 13 14 15 16 17 18 19 20 21 22 23
Aman          █ ▒ █ █ ▒ . . . . . . . . . . . . . . . . . . █
Karan         . . . . . . . . . ▒ █ █ ▒ █ ▒ ▒ ▒ █ █ ▒ ▒ ▒ ▒ .
Neha          . . . . . . . . █ ▒ ▒ ▒ █ █ █ █ █ ▒ █ ▒ █ █ ▒ ▒
Priya         . . . . . . . ▒ █ █ ▒ █ █ █ █ ▒ ▒ ▒ █ █ ▒ █ █ .
Rahul         . . . . . . . . █ ▒ █ █ ▒ █ █ ▒ █ █ █ █ █ █ █ █
Vikas         . . . . . . . . . . ▒ ░ ░ ░ . ▒ ░ ░ █ ░ ░ ▒ . .
Hourly totals: [ 95  72  87  79  74   1   3  32 154 150 163 176 152 197 180 157 177 169
 184 180 167 169 186 169]


## Feature 5 — Top words analysis

The parser builds regular dictionaries for frequency counts. The stop-word list and punctuation handling are visible in the import/constants cell. Media and deleted placeholders never enter word counts.

In [ ]:
def print_word_bars(word_pairs, limit=10):
    pairs = word_pairs[:limit]
    maximum = max((count for word, count in pairs), default=1)
    for word, count in pairs:
        bar_length = max(1, int(22 * count / maximum))
        print(f'  {word:<16} {"█" * bar_length} {count}')

print("GROUP'S TOP TEN WORDS")
print_word_bars(stats['top_words'], 10)
for name in stats['participants']:
    person_words = sorted(stats['word_frequency_by_person'][name].items(), key=lambda pair: (-pair[1], pair[0]))[:5]
    print(f'\n{name} — top five')
    print_word_bars(person_words, 5)


GROUP'S TOP TEN WORDS
  guys             ██████████████████████ 291
  kya              ███████████████████ 252
  da               █████████████████ 227
  what             ███████████████ 210
  here             █████████████ 185
  wait             █████████████ 183
  please           █████████████ 179
  hai              ████████████ 168
  actually         ████████████ 166
  bro              ████████████ 166

Aman — top five
  kya              ██████████████████████ 51
  should           ██████████████████ 43
  da               ████████████████ 39
  plan             ███████████████ 36
  what             ██████████████ 34

Karan — top five
  happened         ██████████████████████ 91
  out              █████████████████████ 88
  still            ████████████████████ 85
  what             █████████████ 56
  anyone           ████████████ 53

Neha — top five
  guys             ██████████████████████ 86
  what             ███████████████████ 78
  here             ███████████████████ 77
  actu

## Feature 6 — Response speed and silent streaks

A response event occurs when the sender changes. Consecutive messages from the same sender do not create additional replies. Silent streaks use the full first-to-last observed date range, including dates when nobody sent a message.

In [ ]:
responders = [(name, item['average_response_minutes'], item['response_count']) for name, item in stats['participant_stats'].items() if item['average_response_minutes'] is not None]
print('Average response time per participant')
for name, minutes, count in sorted(responders, key=lambda item: item[1]):
    print(f'  {name:<14} {minutes:>8.1f} minutes  ({count} reply events)')
if responders:
    print('Fastest replier:', min(responders, key=lambda item: item[1])[0])
    print('Slowest replier:', max(responders, key=lambda item: item[1])[0])
print('\nLONGEST SILENT STREAKS')
for name in sorted(stats['participants'], key=lambda person: (-stats['silent_streaks'][person], person)):
    item = stats['participant_stats'][name]
    dates = f" ({item['silent_start']} to {item['silent_end']})" if item['silent_start'] else ''
    print(f"  {name:<14} {item['longest_silent_days']:>3} days{dates}")


Average response time per participant
  Karan              26.1 minutes  (320 reply events)
  Rahul              28.9 minutes  (528 reply events)
  Neha               30.5 minutes  (405 reply events)
  Vikas              33.7 minutes  (35 reply events)
  Aman               37.5 minutes  (90 reply events)
  Priya              38.6 minutes  (443 reply events)
Fastest replier: Karan
Slowest replier: Priya

LONGEST SILENT STREAKS
  Vikas           14 days (2024-04-07 to 2024-04-20)
  Aman             0 days
  Karan            0 days
  Neha             0 days
  Priya            0 days
  Rahul            0 days


## Feature 7 — Personality archetype detection

Each detection function returns a numeric score and a supporting statistic. The assignment is exclusive: the participant with the strongest top-versus-next score difference claims their highest remaining archetype; ties are resolved alphabetically, then by the listed archetype order. This makes assignments repeatable without participant-name rules. Scores are signals, not clinical labels. The bonus archetype reflects campus food and tea coordination common in Indian college groups.

In [ ]:
# AI-assisted: each score is tied to a quantitative criterion from the brief.
def archetype_spammer(person_records, stats, name):
    """Score the mean back-to-back burst relative to the brief's 3-message rule."""
    bursts = []
    current_burst = 0
    previous_sender = None
    for record in person_records:
        if record["sender"] == name:
            current_burst = current_burst + 1 if previous_sender == name else 1
            bursts.append(current_burst)
        previous_sender = record["sender"]
    average_burst = sum(bursts) / len(bursts) if bursts else 0
    return min(average_burst / 3, 2), average_burst


def archetype_group_mom(person_records, stats, name):
    caring_count = 0
    for record in person_records:
        if record["sender"] == name:
            text = record["text"].lower()
            caring_count += sum(1 for phrase in CARING_PHRASES if phrase in text)
    all_counts = {}
    for person_name in stats:
        all_counts[person_name] = 0
        for record in person_records:
            if record["sender"] == person_name:
                text = record["text"].lower()
                all_counts[person_name] += sum(1 for phrase in CARING_PHRASES if phrase in text)
    max_caring = max(all_counts.values(), default=0)
    return (caring_count / max_caring if max_caring else 0), caring_count


def archetype_night_owl(person_records, stats, name):
    person = [record for record in person_records if record["sender"] == name]
    count = sum(1 for record in person if record["timestamp"].hour >= 23 or record["timestamp"].hour <= 4)
    share = count / len(person) if person else 0
    return share / 0.60, share * 100


def archetype_storyteller(person_records, stats, name):
    average = stats[name]["average_words"]
    return average / 30, average


def archetype_drama_queen(person_records, stats, name):
    person = [record for record in person_records if record["sender"] == name]
    dramatic = 0
    for record in person:
        text = record["text"]
        letters = [character for character in text if character.isalpha()]
        is_caps = len(text) >= 3 and bool(letters) and all(character.isupper() for character in letters)
        if is_caps or text.count("!") >= 2:
            dramatic += 1
    share = dramatic / len(person) if person else 0
    return share / 0.30, share * 100


def archetype_ghost(person_records, stats, name):
    share = stats[name]["silent_percent"] / 100
    return share / 0.60, share * 100


def archetype_comedian(person_records, stats, name):
    person = [record for record in person_records if record["sender"] == name]
    comedy = sum(1 for record in person if any(word in record["text"].lower().split() for word in COMEDY_WORDS))
    share = comedy / len(person) if person else 0
    all_shares = []
    for other in stats:
        other_records = [record for record in person_records if record["sender"] == other]
        count = sum(1 for record in other_records if any(word in record["text"].lower().split() for word in COMEDY_WORDS))
        all_shares.append(count / len(other_records) if other_records else 0)
    highest = max(all_shares, default=0)
    return (share / highest if highest else 0), share * 100


def archetype_question_master(person_records, stats, name):
    person = [record for record in person_records if record["sender"] == name]
    share = sum(1 for record in person if record["text"].rstrip().endswith("?")) / len(person) if person else 0
    return share / 0.25, share * 100


def archetype_canteen_coordinator(person_records, stats, name):
    """Bonus Indian-college archetype: frequent campus food/tea coordination."""
    count = 0
    for record in person_records:
        if record["sender"] == name:
            text = record["text"].lower().split()
            count += sum(1 for word in text if word.strip("!\"#$%&'()*+,-./:;<=>?@[\\]^_`{|}~") in FOOD_WORDS)
    return count, count


def detect_archetypes(person_records, participants, participant_stats):
    functions = (
        ("The Spammer", archetype_spammer), ("The Group Mom", archetype_group_mom),
        ("The Night Owl", archetype_night_owl), ("The Storyteller", archetype_storyteller),
        ("The Drama Queen", archetype_drama_queen), ("The Ghost", archetype_ghost),
        ("The Comedian", archetype_comedian), ("The Question Master", archetype_question_master),
    )
    scores = {}
    for name in participants:
        scores[name] = {}
        for label, function in functions:
            score, evidence = function(person_records, participant_stats, name)
            scores[name][label] = {"score": score, "evidence": evidence}
        bonus_score, bonus_evidence = archetype_canteen_coordinator(person_records, participant_stats, name)
        scores[name]["Bonus: The Canteen Coordinator"] = {"score": bonus_score, "evidence": bonus_evidence}

    # Exclusive, name-agnostic assignment: repeatedly give the strongest
    # remaining archetype to the participant with the strongest available score.
    # Ties use participant alphabetical order, then the stable feature order above.
    available = list(ARCHETYPE_NAMES)
    remaining = set(participants)
    assigned = {}
    while remaining:
        candidates = []
        for name in remaining:
            ranked = sorted(available, key=lambda label: (-scores[name][label]["score"], ARCHETYPE_NAMES.index(label)))
            best = ranked[0]
            next_score = scores[name][ranked[1]]["score"] if len(ranked) > 1 else 0
            candidates.append((-(scores[name][best]["score"] - next_score), -scores[name][best]["score"], name, best))
        _, _, chosen_name, chosen_label = min(candidates)
        assigned[chosen_name] = {"primary": chosen_label, **scores[chosen_name][chosen_label]}
        available.remove(chosen_label)
        remaining.remove(chosen_name)
    for name in participants:
        assigned[name]["scores"] = {label: detail["score"] for label, detail in scores[name].items() if label in ARCHETYPE_NAMES}
        assigned[name]["bonus"] = scores[name]["Bonus: The Canteen Coordinator"]
    return assigned

stats['archetypes'] = detect_archetypes(participant_records, stats['participants'], stats['participant_stats'])
for name in stats['participants']:
    assignment = stats['archetypes'][name]
    print(f"{name:<14} → {assignment['primary']:<20} evidence={assignment['evidence']}")
    print('  Scores:', ', '.join(f"{label}: {score:.2f}" for label, score in assignment['scores'].items()))
    print('  Bonus — The Canteen Coordinator:', assignment['bonus']['evidence'], 'food / tea terms')


Aman           → The Night Owl        evidence=99.15966386554622
  Scores: The Spammer: 1.42, The Group Mom: 0.18, The Night Owl: 1.65, The Storyteller: 0.14, The Drama Queen: 0.00, The Ghost: 0.00, The Comedian: 0.00, The Question Master: 1.37
  Bonus — The Canteen Coordinator: 32 food / tea terms
Karan          → The Storyteller      evidence=10.974025974025974
  Scores: The Spammer: 0.40, The Group Mom: 0.07, The Night Owl: 0.00, The Storyteller: 0.37, The Drama Queen: 0.00, The Ghost: 0.00, The Comedian: 0.00, The Question Master: 0.15
  Bonus — The Canteen Coordinator: 18 food / tea terms
Neha           → The Drama Queen      evidence=49.411764705882355
  Scores: The Spammer: 0.42, The Group Mom: 0.08, The Night Owl: 0.08, The Storyteller: 0.16, The Drama Queen: 1.65, The Ghost: 0.00, The Comedian: 0.00, The Question Master: 0.15
  Bonus — The Canteen Coordinator: 22 food / tea terms
Priya          → The Group Mom        evidence=238
  Scores: The Spammer: 0.44, The Group Mom: 1.0

## Feature 8 — Final GroupDNA report

The next cell combines the computed results into a terminal-friendly layout suitable for a screenshot.

In [ ]:
LINE = '=' * 78
print(LINE)
print(f"  GROUPDNA REPORT — {stats['group_name']}")
print(f"  {stats['duration_days']} days · {stats['total_messages']:,} participant entries · {len(stats['participants'])} members")
print(LINE)
print(f"  Period: {stats['start'][:10]} to {stats['end'][:10]}")
print(f"  Busiest day: {stats['busiest_day']} ({stats['busiest_day_count']} messages)")
print(f"  Busiest hour: {stats['busiest_hour']:02d}:00–{(stats['busiest_hour']+1)%24:02d}:00")
print(f"  Entry details: {stats['system_messages']} system · {stats['media_messages']} media · {stats['deleted_messages']} deleted · {stats['analyzable_messages']} analyzable text")
print('\n  MESSAGES PER PARTICIPANT')
for name in sorted(stats['participants'], key=lambda person: (-stats['participant_stats'][person]['messages'], person)):
    person = stats['participant_stats'][name]
    bar = '█' * max(1, int(20 * person['messages'] / max(1, stats['total_messages'])))
    print(f"    {name:<14} {bar:<20} {person['messages']:>5,} ({person['contribution_percent']:5.1f}%)")
print('\n  ACTIVITY HEATMAP (all 24 hours)')
print('           ' + ' '.join(f'{h:02d}' for h in range(24)))
for idx, name in enumerate(stats['participants']):
    values = activity_matrix[idx]
    top = int(np.max(values))
    shades = []
    for value in values:
        ratio = value / top if top else 0
        shades.append('.' if value == 0 or ratio < .25 else '░' if ratio < .5 else '▒' if ratio < .75 else '█')
    print(f"    {name:<10} {' '.join(shades)}")
print('\n  TOP WORDS')
print_word_bars(stats['top_words'], 10)
print('\n  TOP FIVE WORDS PER PARTICIPANT')
for name in stats['participants']:
    print(f"    {name}:")
    person_words = sorted(stats['word_frequency_by_person'][name].items(), key=lambda pair: (-pair[1], pair[0]))[:5]
    print_word_bars(person_words, 5)
print('\n  RESPONSE TIME AND SILENT STREAKS')
for name in sorted(stats['participants'], key=lambda person: (stats['participant_stats'][person]['average_response_minutes'] is None, stats['participant_stats'][person]['average_response_minutes'] or 0)):
    person = stats['participant_stats'][name]
    average = 'no reply events' if person['average_response_minutes'] is None else f"{person['average_response_minutes']:.1f} min"
    print(f"    {name:<14} avg reply {average:<16} longest silent streak {person['longest_silent_days']:>2} days")
    if person['silent_start']:
        print(f"      Silent dates: {person['silent_start']} to {person['silent_end']}")
print('\n  PERSONALITY ARCHETYPES AND BONUS SIGNAL')
for name in stats['participants']:
    assignment = stats['archetypes'][name]
    evidence = assignment['evidence']
    evidence_text = f"{evidence:.1f}" if isinstance(evidence, float) else str(evidence)
    print(f"    {name:<14} → {assignment['primary']:<20} (evidence: {evidence_text})")
    print(f"      Canteen Coordinator: {assignment['bonus']['evidence']} food / tea terms")
print(LINE)
print('  Generated by GroupDNA · Python + NumPy')
print(LINE)


  GROUPDNA REPORT — Hostel Bois 4ever
  60 days · 3,173 participant entries · 6 members
  Period: 2024-04-01 to 2024-05-30
  Busiest day: 2024-04-19 (66 messages)
  Busiest hour: 13:00–14:00
  Entry details: 4 system · 30 media · 15 deleted · 3128 analyzable text

  MESSAGES PER PARTICIPANT
    Rahul          ███████              1,177 ( 37.1%)
    Priya          ███                    580 ( 18.3%)
    Neha           ███                    510 ( 16.1%)
    Aman           ███                    476 ( 15.0%)
    Karan          ██                     389 ( 12.3%)
    Vikas          █                       41 (  1.3%)

  ACTIVITY HEATMAP (all 24 hours)
           00 01 02 03 04 05 06 07 08 09 10 11 12 13 14 15 16 17 18 19 20 21 22 23
    Aman       █ ▒ █ █ ▒ . . . . . . . . . . . . . . . . . . █
    Karan      . . . . . . . . . ▒ █ █ ▒ █ ▒ ▒ ▒ █ █ ▒ ▒ ▒ ▒ .
    Neha       . . . . . . . . █ ▒ ▒ ▒ █ █ █ █ █ ▒ █ ▒ █ █ ▒ ▒
    Priya      . . . . . . . ▒ █ █ ▒ █ █ █ █ ▒ ▒ ▒ █ █ ▒ █ █ .
    Rahu

## 15. Additional analysis and observations

The dataset is synthetic. The notebook reports observed counts directly and does not alter them to match the brief's original reference values. Compare those reference values only when the official dataset is supplied.

In [ ]:
print('Observed total entries:', len(records))
print('Observed participant entries:', stats['total_messages'])
print('Observed text messages:', stats['analyzable_messages'])
print('Observed special-entry counts:', parser_counters)
print('Observed participant message counts:', {name: stats['participant_stats'][name]['messages'] for name in stats['participants']})


Observed total entries: 3177
Observed participant entries: 3173
Observed text messages: 3128
Observed special-entry counts: {'system': 4, 'media': 30, 'deleted': 15, 'empty': 0, 'continuation': 0}
Observed participant message counts: {'Aman': 476, 'Karan': 389, 'Neha': 510, 'Priya': 580, 'Rahul': 1177, 'Vikas': 41}


## 16. Testing and validation

These small checks cover empty input, one and multiple participants, system/media/deleted entries, multi-line continuation, chronological aggregation, zero activity, and exclusive archetype assignments. They use in-memory line lists so no extra testing library is needed.

In [ ]:
def check(condition, label):
    print(('PASS' if condition else 'REVIEW') + ' — ' + label)

empty_records, empty_counts = parse_chat_lines([])
empty_stats = build_statistics(empty_records)
check(len(empty_records) == 0 and empty_stats['total_messages'] == 0, 'Empty dataset')
one_records, _ = parse_chat_lines(['01/04/24, 10:00 - Riya: hello\n', '01/04/24, 10:05 - Riya: okay\n'])
one_stats = build_statistics(one_records)
check(len(one_stats['participants']) == 1 and one_stats['total_messages'] == 2, 'Single participant')
check(one_stats['participant_stats']['Riya']['average_response_minutes'] is None, 'No-response participant handled without division by zero')
check(one_stats['participant_stats']['Riya']['longest_silent_days'] == 0, 'Zero silent days represented correctly')
edge_lines = ['01/04/24, 10:00 - System changed group description\n', '01/04/24, 10:02 - Riya: hello\n', 'continued line\n', '01/04/24, 10:04 - Sam: <Media omitted>\n', '01/04/24, 10:05 - Sam: This message was deleted\n', '\n']
edge_records, edge_counts = parse_chat_lines(edge_lines)
edge_stats = build_statistics(edge_records)
check(edge_counts['system'] == 1, 'System message counted separately')
check(edge_counts['media'] == 1 and edge_stats['participant_stats']['Sam']['media'] == 1, 'Media retained as activity')
check(edge_counts['deleted'] == 1 and edge_stats['participant_stats']['Sam']['deleted'] == 1, 'Deleted entry separately counted')
check(edge_records[1]['text'] == 'hello\ncontinued line', 'Multi-line continuation appended')
check(edge_counts['empty'] == 1, 'Empty line skipped')
check(edge_stats['total_messages'] == 3 and edge_stats['analyzable_messages'] == 1, 'Participant activity separated from text messages')
unsorted, _ = parse_chat_lines(['02/04/24, 10:00 - Sam: later\n', '01/04/24, 10:00 - Riya: earlier\n'])
ordered_stats = build_statistics(unsorted)
check(ordered_stats['start'].startswith('2024-04-01') and ordered_stats['end'].startswith('2024-04-02'), 'Timestamp parsing and chronological aggregation')
check(activity_matrix.shape[1] == 24 and np.sum(activity_matrix) == stats['total_messages'], 'NumPy dimensions and row totals')
assignments = [item['primary'] for item in stats['archetypes'].values()]
check(len(assignments) == len(stats['participants']) and len(set(assignments)) == len(assignments), 'Exactly one exclusive archetype per participant')
check(all(len(item['scores']) == 8 for item in stats['archetypes'].values()), 'All eight archetype scores calculated')
check(stats['total_entries'] == len(records), 'End-to-end file execution')


PASS — Empty dataset
PASS — Single participant
PASS — No-response participant handled without division by zero
PASS — Zero silent days represented correctly
PASS — System message counted separately
PASS — Media retained as activity
PASS — Deleted entry separately counted
PASS — Multi-line continuation appended
PASS — Empty line skipped
PASS — Participant activity separated from text messages
PASS — Timestamp parsing and chronological aggregation
PASS — NumPy dimensions and row totals
PASS — Exactly one exclusive archetype per participant
PASS — All eight archetype scores calculated
PASS — End-to-end file execution


## 17. Project conclusion

GroupDNA reads a WhatsApp export, preserves message activity while separating non-text entries, and summarizes group rhythms, vocabulary, reply gaps, silent streaks, and behaviour signals. NumPy powers a dynamic 24-hour participant matrix. Results are descriptive and depend on the supplied export, tokenization choices, and the stated scoring rules.

## 18. Student reflection

Replace this paragraph with your own reflection before submission: What was hardest to parse or analyze? What would you change after reviewing the results? Which archetype did you get on your own chat, if you chose to run the optional private analysis? Never submit or publish private chat text. The supplied practice dataset is synthetic and should be described as such.